# Lost in Transcription — Spanish–English

Independent Spanish–English workflow. Do not mix its data, weights, glossary, TensorBoard logs, Hub repository, or submission archive with the Indonesian–Javanese or Spanish–Nahuatl tracks.

In [ ]:
# Downloads only when Bangor is absent; the script reads MDC_API_KEY from ROOT / '.env'.
if not (BANGOR_DIR.exists() and any(BANGOR_DIR.rglob('*'))):
    subprocess.run([sys.executable, str(ROOT / 'spanish_english' / 'download_bangor_miami.py'), '--download'], check=True, env=os.environ.copy())
else:
    print(f'Reusing {BANGOR_DIR}')


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
from getpass import getpass

def find_project_root() -> Path:
    required = ("requirements-runpod.txt", "src", "spanish_english")
    configured = os.environ.get('LIT_ROOT')
    candidates = ([Path(configured)] if configured else []) + [Path.cwd(), *Path.cwd().parents]

    for candidate in candidates:
        if all((candidate / item).exists() for item in required):
            return candidate.resolve()

    raise FileNotFoundError(
        "Could not locate the project root. Open the notebook from the repository "
        "or set ROOT manually."
    )

ROOT = find_project_root()

if not ROOT.is_dir():
    raise FileNotFoundError(f'Update ROOT: {ROOT}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', '--ignore-installed', 'blinker'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', '--upgrade-strategy', 'eager', '-r', str(ROOT / 'requirements-runpod.txt')], check=True)
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'verify_runpod.py')], check=True)
sys.path[:0] = [str(ROOT / 'src'), str(ROOT)]
SPEN_DATA = ROOT / 'data' / 'spanish_english'
SPEN_DEV_DIR = SPEN_DATA / 'development'
BANGOR_DIR = SPEN_DATA / 'bangor_miami'
ARTIFACTS = ROOT / 'artifacts' / 'spen'
TENSORBOARD_ROOT = ARTIFACTS / 'tensorboard'
ARTIFACTS.mkdir(parents=True, exist_ok=True); TENSORBOARD_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
from spanish_english import prepare
from lit_jv import prompts
from lit_jv.decode import DecodeConfig, FallbackConfig, Qwen3AsrDecoder, build_backend
from lit_jv.scoring import wer_report
from lit_jv.train_lora import AugmentSpec, LoraSpec, StageSpec, TrainSpec, merge_and_save, train

dev_rows = prepare.load_competition_dev(SPEN_DEV_DIR)
bangor_rows = prepare.load_bangor_miami(BANGOR_DIR)
dev_train, dev_eval = prepare.holdout_speakers(dev_rows, count=1, seed=2026)
selection_train = bangor_rows + dev_train
terms = prepare.glossary(selection_train)
(ARTIFACTS / 'glossary.json').write_text(json.dumps(terms, ensure_ascii=False, indent=2), encoding='utf-8')
print({'bangor_train': len(bangor_rows), 'dev_train': len(dev_train), 'dev_eval': len(dev_eval), 'glossary': len(terms)})

prompt = prompts.PromptConfig(prose='codeswitch_spen', style_tag=prompts.STYLE_VOICE_NOTE_SP_EN, glossary=tuple(terms), max_glossary_terms=128, force_language='Spanish')
def score(model_path, rows, name):
    backend = build_backend(model_path, prefer='vllm', log=print)
    decoder = Qwen3AsrDecoder(backend, fallback=FallbackConfig(enabled=False), log=print)
    try:
        hypotheses = decoder.decode_pass(rows, DecodeConfig(prompt=prompt, name=name, temperature=0.0), chunk_size=16)
        return wer_report([row['raw_text'] for row in rows], [hyp.text for hyp in hypotheses])
    finally:
        decoder.close()
baseline = score('Qwen/Qwen3-ASR-1.7B', dev_eval, 'qwen-zero-shot')
print(baseline.as_row('qwen-zero-shot'))


In [ ]:
spec = TrainSpec(
    model_path='Qwen/Qwen3-ASR-1.7B', output_dir=str(ARTIFACTS / 'lora'), target_language='Spanish',
    lora=LoraSpec(r=16, alpha=32, adapt_text_decoder=True, adapt_audio_encoder=False),
    augment=AugmentSpec(enabled=False), prompt=prompt, eval_steps=0, wer_eval_steps=10,
    wer_eval_clips=len(dev_eval), wer_eval_batch_size=4, tensorboard_dir=str(TENSORBOARD_ROOT / 'selection'),
    report_to=('tensorboard',), run_name='spen-qwen3asr',
)
stages = (StageSpec('s1-bangor', {'bangor_miami': 1.0}, epochs=1.0, learning_rate=8e-5, batch_size=4, grad_accumulation=8), StageSpec('s2-dev', {'spen_dev': 1.0}, epochs=3.0, learning_rate=3e-5, batch_size=4, grad_accumulation=8))
selected = train(selection_train, spec, stages=stages, save_adapter_as='selected', eval_rows=dev_eval)
merged = merge_and_save(spec, ARTIFACTS / 'lora' / 'selected', ARTIFACTS / 'merged')
selected_dev = score(merged, dev_eval, 'selected-dev')
assert selected_dev.wer < baseline.wer, 'Reject: selected adapter did not beat zero-shot.'
print(selected_dev.as_row('selected-dev'))


In [ ]:
final_spec = TrainSpec(**{**spec.__dict__, 'output_dir': str(ARTIFACTS / 'final-lora'), 'tensorboard_dir': str(TENSORBOARD_ROOT / 'final'), 'wer_eval_steps': 0})
train(bangor_rows + dev_rows, final_spec, stages=stages, save_adapter_as='final', eval_rows=None)
final_merged = merge_and_save(final_spec, ARTIFACTS / 'final-lora' / 'final', ARTIFACTS / 'final-merged')
subprocess.run([sys.executable, str(ROOT / 'spanish_english' / 'package_submission.py'), '--model', str(final_merged), '--glossary', str(ARTIFACTS / 'glossary.json')], check=True)
SUBMISSION_ZIP = ARTIFACTS / 'submission.zip'
assert SUBMISSION_ZIP.is_file()
print('TensorBoard:', TENSORBOARD_ROOT)
print('Jupyter: %load_ext tensorboard; %tensorboard --logdir ' + str(TENSORBOARD_ROOT))


## Publish this track only

Set `SPEN_HF_REPO` if you want a custom Hub destination. The default is a new repository under the authenticated account. This uploads this track's final weights, ZIP, and TensorBoard events only.

In [ ]:
from huggingface_hub import HfApi
if not os.environ.get('HF_TOKEN'): os.environ['HF_TOKEN'] = getpass('Hugging Face write token (not saved): ')
api = HfApi(token=os.environ['HF_TOKEN']); account = api.whoami()['name']
REPO_ID = os.environ.get('SPEN_HF_REPO', f'{account}/lost-in-transcription-spanish-english')
api.create_repo(repo_id=REPO_ID, repo_type='model', private=os.environ.get('SPEN_HF_PRIVATE', '0') == '1', exist_ok=True)
api.upload_folder(repo_id=REPO_ID, repo_type='model', folder_path=str(final_merged), path_in_repo='model', commit_message='Upload final Spanish-English model weights')
api.upload_file(path_or_fileobj=str(SUBMISSION_ZIP), path_in_repo='submission.zip', repo_id=REPO_ID, repo_type='model', commit_message='Upload Spanish-English submission archive')
api.upload_folder(repo_id=REPO_ID, repo_type='model', folder_path=str(TENSORBOARD_ROOT), path_in_repo='runs', commit_message='Upload TensorBoard curves')
print(f'https://huggingface.co/{REPO_ID}')
